# Introduction to CrissCross: Extracting Spectra from Crowded Regions
This notebook provides two end-to-end examples of how to extract a single HETG spectrum from a crowded region and run CrissCross to handle confusion. Example 

1.  covers a single obsID in depth and 
2.  demonstrates how to run CrissCross on multiple obsIDs and combine the resulting spectra. 

More information and examples on `crisscross` and its helper function `clean_spec` can be found on their respective ahelp pages (e.g., type `ahelp crisscross` and `ahelp clean_spec` on the command line in CIAO).

## CrissCross Summary
CrissCross and its helper tool clean_spec allow users to analyze Chandra High Energy Transmission Grating (HETG) 
spectra that are extracted from a field of view with multiple other astrophysical X-ray sources. The HETG 
instrument will disperse events onto the ACIS CCDs from all sources in a field of view. If there are several 
sufficiently-bright X-ray sources then there is a potential for 'confusion' to occur. Confusion is a term used 
here for special scenarios where [standard HETG 
spectral extractions](https://cxc.harvard.edu/ciao/threads/gspec.html) of a source may erroneously assign events (counts) from a different astrophysical 
source to the extracted source. This can result in events from an unrelated astrophysical 
source 'confusing' the spectrum of an extracted source.

CrissCross is used to identify when spectral confusion occurs for a list of input sources (or a single source) 
and generates 'confusion tables' which identify the wavelength in each source's spectrum that are likely to 
include X-ray events from unrelated field astrophysical sources. The user can then choose to remove all events 
from the source spectrum in the wavelength range where this confusion occurs. The CrissCross helper program 
clean_spec uses the CrissCross output confusion tables and input HETG PHA spectra ```````to generate 'cleaned' spectra 
(and ARFs) where these events are removed. CrissCross is especially useful for crowded X-ray fields with multiple 
observations such as stellar clusters.

## Running CrissCross

In [ ]:
import os
import glob
import shutil

import numpy as np
import matplotlib.pyplot as plt

from ciao_contrib.runtool import *
from sherpa.astro.ui import *

## (1) Run CrissCross for a single source on a single obsID

#### Download obsID 3, a crowded HETG observation of a pre-main sequence stellar cluster.

In [ ]:
os.system('download_chandra_obsid 3')

#### We will extract the spectrum of a single source in the field of view using chandra_repro and the source coordinates. COUP 394 is a young, pre-main sequence star whose X-ray emission is generated by a hot corona. The commands below will produce only responses for the +1/-1 HEG/MEG orders. Users that wish to generate all responses should use the 'tg_orders' command with chandra_repro. 

#### Users that wish to see spectra from other sources in the field of view can modify the variables below before running the remaining cells

In [ ]:
#SRC NAME -- COUP 394
#83.79475306,-5.39575306,393

src_RA = 83.79475306
src_DEC = -5.39575306
obsid = 3
src_root = 'COUP_394'

In [ ]:
#run chandra repro using the zeroth order source location for COUP 394

chandra_repro.punlearn()
chandra_repro.indir = obsid
chandra_repro.root = src_root
chandra_repro.tg_zo_position = f'{src_RA},{src_DEC}'
chandra_repro.clobber=False

print(chandra_repro)

a = chandra_repro()

#### Lets take a look at the source's spectra using Sherpa. We can use the new sherpa-contrib function 'load_gratings_pha2' to identify and load the pipeline response files (ARFs and RMFs) use information in the PHA2 spectrum header. Since the PHA2 file has all spectra data for all 12 orders, sherpa data ids 1-12 will be used. If the directory does not include all 12 HETG responses then only those identified will be loaded. 

In [ ]:
clean() #run clean in case you want to refresh this cell in a notebook 

original_pha2 = f'{obsid}/repro/{src_root}_repro_pha2.fits'
load_gratings_pha2(pha2_file=original_pha2, use_errors=True)

#### Now that the data are loaded, we can plot the brightest orders (HEG-1, HEG+1, MEG-1, MEG+1). Since there is not much data past 20 Angstrom for this source we will limit the X-axis range.

In [ ]:
#to view all four orders at once (heg +1/-1 and meg+1/-1)
spec_ids = [3,4,9,10]

#each id must be set to the 'wave' analysis to plot in angstrom (instead of keV)
for i in spec_ids:
    set_analysis(i,'wave')

bin_num=3
for i in spec_ids:
    group_width(i, bin_num)

#create an arm label for the subplot names
arm_label = ['heg-1','heg+1','meg-1','meg+1']

plt.figure(figsize=(9, 5))
plot("data", 3, "data", 4, "data", 9, "data", 10, yerrorbars=False, color='red', linestyle='-', marker=None)

plt.subplots_adjust(left=0, hspace=0.5, wspace=0.3)
axes = plt.gcf().axes

# set the same x-limits for all panels:
#for ax in axes:
for ax, arm in zip(axes,arm_label):
    ax.set_xlim(0, 20)
    ax.set_title(arm)

#### We can also plot the spectral response for the source using the ARF files. This shows the sensitivity of the detector at the extracted spectral location.

In [ ]:
#we can also plot the ARFs for each order
arm_label_arf = ['heg-1 response','heg+1 response','meg-1 response','meg+1 response']

plt.figure(figsize=(9, 5))
plot("arf", 3, "arf", 4, "arf", 9, "arf", 10, color='red')

axes = plt.gcf().axes
for ax, arm in zip(axes,arm_label_arf):
    ax.set_xlim(0, 20)
    ax.set_title(arm)

plt.subplots_adjust(left=0, hspace=0.5, wspace=0.3)

#### The event file can be viewed via ciao's ds9 commands:

In [ ]:
os.system(f'ds9 {obsid}/repro/{src_root}_repro_evt2.fits -scale log -bin factor 4 &')

#### The extracted source is in a crowded field and confusion should be assessed. In other words, the dispersed spectrum of COUP 394 (green ds9 region-box if viewing the event file) intersects with many sources and other dispersed spectra.  To identify potential confusion we'll run CrissCross on the observation. This requires an input list of all X-ray sources potentially in the field of view. 

#### I have included such a list for this tutorial (see download at top) but users will have to produce their own for other fields. Most of the Chandra HETG observations with more than a few sources in the field of view have been previously observed with Chandra imaging. Running CIAO's wavdetect on an archival non-HETG imaging observaiton is a good way to produce an easy source list. Otherwise, users will have to use another wavelength (telescope) or identify the 0th order sources in an HETG observation by eye and manually create a list.

#### An example of how to create a crisscross source list using wavdetect can be found on the crisscross ahelp page.

In [ ]:
help(run_crisscross)
#os.system('ahelp crisscross') #NEEDS TO BE UPDATED BEFORE RELEASE

#### If you have a source list (the main_list par) then all you need for this tutorial is the evt2 file and RA/DEC of your HETG-extracted source. CrissCross will identify if any of the sources in the source list produce erroneous counts in the extracted source COUP 394. Depending on your computer and the number of sources in the source list, CrissCross could potentially take a few minutes to run.

In [ ]:
evt2_path = f'{obsid}/repro/{src_root}_repro_evt2.fits'

run_crisscross(evt2_file=evt2_path, cc_outdir='cc_tutorial', main_list='input_files/full_coup_src_list.tsv',  single_src_pos=f'{src_RA},{src_DEC}', single_src_root=src_root, clobber_par=False) #NEEDS TO BE UPDATED BEFORE RELEASE

#### CrissCross has finished and produced the relevant cleaning table for COUP 394. We can view the identified confusion for each arm with dmlist (or any fits table reader). Here, we only print the most relevant columns but the full table can be viewed as well.

In [ ]:
cc_table_coup = f'cc_tutorial/output_dir_obsid_3/confusion_output_files/table_fits_data/confused_{src_root}_consolidated_obsID_3.fits'

#view the sources of confusion in the COUP 394 MEG arm
dmlist.punlearn
dmlist.infile = f'{cc_table_coup}[cols confuser_srcid, grating_type, order, confusion_type, confusion_wave, wave_low, wave_high, flag][flag=confused, order=-1,+1, grating_type=meg]'
dmlist.opt='data'

dmlist()

In [ ]:
#view the sources of confusion in the COUP 394 HEG arm
dmlist.punlearn
#dmlist.infile = f'{cc_table_coup}[flag=confused, order=-1,+1, grating_type=heg]'
dmlist.infile = f'{cc_table_coup}[cols confuser_srcid, grating_type, order, confusion_type, confusion_wave, wave_low, wave_high, flag][flag=confused, order=-1,+1, grating_type=heg]'
dmlist.opt='data'
dmlist.rows='1:50'

dmlist()

#### We can see that CrissCross identified some confusion (flag column 'confused'). The 'wave_low' and 'wave_high' columns indicate the wavelength range recommended to be ignored due to the liklihood of HETG confusion by source 'confuser_srcid'. The 'confusion_type' column shows which of the three primary sources of confusion (point source, spectral intersection or arm confusion) has been identified. For more information on each of these types of confusion, see the crisscross ahelp page. 

#### Let's clean the confused portions of the spectrum and responses using the CrissCross helper CIAO function clean_spec. NOTE: clean_spec removes ALL counts in the confused regions and not just counts from the confusing source. However, CrissCross has many tunable parameters to allow some level of potential confusion through based on the signal-to-noise ratio of the HETG source events compared to the confused events in the confused bandpass. If you find CrissCross is unnecessarily removing counts then please re-run with lower confusion threshold parameters. See the crisscross ahelp page for more info.

In [ ]:
help(clean_spec)
#os.system(ahelp clean_spec) #NEEDS TO BE UPDATED BEFORE RELEASE

#### clean_spec will attempt to identify the relevant response files when provided with an HETG PHA2 file. This is important because clean_spec also zeros out the ARF for each source of confusion. The clean_spec tool creates a new 'cleaned' PHA2 file and associated ARFs and does not modify the original files. In this example, only 4 ARFs and spectra are cleaned because we only created 4 response files when running chandra repro (heg-1,+1, meg-1,+1). Even though only four orders have been cleaned, the new PHA2 file still contains the other 8 uncleaned (original) orders.

In [ ]:
clean_spec(cc_table=cc_table_coup, pha_file = original_pha2, spec_root=src_root, arf_file=None, resp_dir=None, clobber=False)

#### clean_spec produced a new PHA2 file and four new ARF files. So lets load those into sherpa and take a look. We'll load them starting with ID 13 so we don't overwrite the previously loaded (uncleaned) spectra. We will find the responses again and load the new cleaned ARFs and the original RMFs. Note, the RMF files are not affected by confusion and thus we will use the original RMFs produced, in this case, with chandra_repro.


In [ ]:
cleaned_pha2 = f'{src_root}_obsid_{obsid}_cleaned.pha2'
load_gratings_pha2(pha2_file=cleaned_pha2, rmf_dir=f'{obsid}/repro/tg', use_errors=True, dataset_id_start=13)

#### Now that we have cleaned spectra and responses, we can view the cleaned spectra as we did before but this time overplot them on the original. This will demonstrate what CrissCross and clean_spec have done.

In [ ]:
#the new sherpa ids for the cleaned heg-1, heg+1, meg-1 and meg+1 spectra
spec_ids_clean = [15,16,21,22]

#set the analysis space to wavelength so it plots in Angstroms.
for i in spec_ids_clean:
    set_analysis(i,'wave')

#bin num was set in earlier cell and should be the same here
for i in spec_ids_clean:
    group_width(i, bin_num)

plt.figure(figsize=(9, 5))

#plot the original (confused) spectra in red
plot("data", 3, "data", 4, "data", 9, "data", 10, color='red',label=['confused','confused','confused','confused'], yerrorbars=False, linestyle='-', marker=None)

#overplot the new 'cleaned' spectra in blue
plot("data", 15, "data", 16, "data", 21, "data", 22, color='blue',label=['cleaned','cleaned','cleaned','cleaned'], overplot=True, yerrorbars=False, linestyle='-', marker=None)

# set the same x-limits for all panels:
axes = plt.gcf().axes
for ax, arm in zip(axes,arm_label):
    ax.set_xlim(0,20)
    ax.set_title(arm)
    ax.legend()

#label the figure with color
fig = plt.gcf()
fig.text(0.30, 0.95, "Confused",  ha='left',  va='center', color='red',  fontsize=16)
fig.text(0.425, .95, "vs",    va='center', color='black',  fontsize=16)
fig.text(0.565, .95, "Cleaned", ha='right', va='center', color='blue', fontsize=16)
plt.draw()
plt.subplots_adjust(left=0, hspace=0.5, wspace=0.3)

#### We can directly see the identified regions of confusion where the flux (blue) goes to zero compared to the red spectra. We can also make the similar comparison with the original (confused) and cleaned arfs:

In [ ]:
plt.figure(figsize=(9, 5))
#original ARFs
plot("arf", 3, "arf", 4, "arf", 9, "arf", 10, label=['confused','confused','confused','confused'], color='red')
#cleaned ARFs
plot("arf", 15, "arf", 16, "arf", 21, "arf", 22, label=['cleaned','cleaned','cleaned','cleaned'], color='blue', overplot=True)

axes = plt.gcf().axes
for ax, arm in zip(axes,arm_label_arf):
    ax.set_xlim(0, 20)
    ax.set_title(arm)

plt.subplots_adjust(left=0, hspace=0.5, wspace=0.3)

#### We can see several portions of each spectrum have been identified to have events associated with other sources in the field assigned to our extracted COUP 394 source (e.g., confusion). Notice what would have appeared as emission lines now are identfied as emission from a different source. The small ranges of confusion are typically associated with point source and spectral intersection whereas the large bandpass of confusion are typically the result of arm confusion (e.g., another bright source falling on the dispersed arm of the extract source.)

#### We can double check the confusion tables to see specific instances of confusion and the source causing them. Let's look at confusion the MEG arm data.

In [ ]:
dmlist.punlearn
dmlist.infile = f'{cc_table_coup}[cols confuser_srcid, grating_type, order, confusion_type, confusion_wave, wave_low, wave_high, flag][flag=confused, order=-1,+1, grating_type=meg]'
dmlist.opt='data'
dmlist.rows=30

dmlist()

#### From the table, it appears that the MEG+1 wavelength range of ~8-9.5 A is affected by two sources of confusion. A 0th order point source (COUP 187) landed right on the dispersed spectrum and the dispersed spectrum of COUP 1390 also intersected with the extracted MEG+1 spectrum.

#### Lets take a closer look at the events assigned to the HEG or MEG spectrum to visually assess how well CrissCross identified confusion. This plotting method only works on evt2 files made for the extracted source of interest which in this case is COUP 394. Here we can see what information is available in a typical HETG evt2 file.

In [ ]:
#use pycrates to read in the event file and inspect the columns
from pycrates import read_file

evt2_file = f'{obsid}/repro/{src_root}_repro_evt2.fits'
evt_data = read_file(evt2_file)
evt_data.get_colnames()

#### We will save the relevant HETG event values to new arrays for plotting. HETG event files contain extra columns for events associated with the extracted spectrum compared to ACIS imaging observations.

In [ ]:
#the HETG wavelength assigned to each event
tg_lam = evt_data.tg_lam.values
#the cross-dispersion distance (distance from 0th order) of each event
tg_d = evt_data.rd.tg_d.values
#the order (e.g., +1,-1) assigned to each event
tg_m = evt_data.tg_m.values
#the order times the wavelength for each event
tg_mlam = evt_data.tg_mlam.values
#the ACIS CCD-derived energy of the event
energy = evt_data.energy.values

In [ ]:
def filter_evt_crate(evt_crate, order_val, arm_par, tgd_par):
    """
    Function for filtering a crate evt file to return the event rows that satisfy the HETG order, arm type (meg/heg) and tg_d value conditions. 
    """
    crate_elements = np.where( (evt_crate.tg_m.values == order_val) & (evt_crate.tg_part.values == arm_par) & (np.abs(evt_crate.rd.tg_d.values) < tgd_par) )

    return(crate_elements)

### Here we filter the event file to retrieve order- and arm-specific event information for plotting.

In [ ]:
#create the filters for plotting only the relevant orders and arm type
meg_p1 = filter_evt_crate(evt_data, 1, 2, 8E-3)
meg_m1 = filter_evt_crate(evt_data,-1, 2, 8E-3)
heg_p1 = filter_evt_crate(evt_data, 1, 1, 8E-3)
heg_m1 = filter_evt_crate(evt_data, -1, 1, 8E-3)

fig, ax = plt.subplots(
    nrows=1, ncols=2,                # 2 rows, 2 columns
    figsize=(16, 5),                # optional: figure size in inches
)

#MEG PLOT
ax[0].scatter(-1*tg_lam[meg_m1], tg_d[meg_m1], s=1, color='green')
ax[0].scatter(tg_lam[meg_p1], tg_d[meg_p1], s=1, color='blue')
ax[0].plot([-30,30], [6.4E-4, 6.4E-4], color='orange') #default extraction width for HETG spectra
ax[0].plot([-30,30], [-6.4E-4, -6.4E-4], color='orange')
ax[0].set_title('MEG -1 and +1')

#HEG PLOT
ax[1].scatter(-1*tg_lam[heg_m1], tg_d[heg_m1], s=1, color='green')
ax[1].scatter(tg_lam[heg_p1], tg_d[heg_p1], s=1, color='blue')
ax[1].set_title('HEG -1 and +1')
ax[1].plot([-15,15], [6.4E-4, 6.4E-4], color='orange')
ax[1].plot([-15,15], [-6.4E-4, -6.4E-4], color='orange')

for i in ax:
    i.set_xlabel('m * Wavelength [Angstrom]')
    i.set_ylabel('Cross Dispersion Angle [deg]')

#### The above plot shows the MEG (left) and HEG (right) events associated with the CIAO extraction of COUP 394 source. Specifically, the events within the orange box will be those that are included in the PHA2 spectrum. The X-axis denotes the order (1 or -1) * wavelength and the y-axis denotes the cross dispersion angle (tg_d) in degrees. This y-axis value is essentially denotes the counts above and below the 'line' of the dispersed spectrum.

#### A single unconfused source should only show a 'bar' of events within the orange box. On the MEG+1 plot (right, blue) you can see a strong vertical line at about 9 Angstroms which is the location of a 0th order (point source) field star that happend to fall on the dispersed spectrum of our extracted source. The vertical nature of this feature is due to the size of the PSF of that source and many of those 0th-order events were the appropriate energy (e.g., 9 A) to erroneously be assigned to the MEG+1 spectrum of COUP 394. Other fainter 0th-order confusing sources can be seen in MEG-1 at approximately -11 and -5 A. The two large horizontal features shown in the MEG plot at y\~0.008 and -0.006 represent the dispersed spectra of other stars in the field which are close to the extracted source. However, only events with a small range in cross dispersion angle (\~+/- 6.3E-4) and assigned to the extracted source with CIAO tools.

#### The HEG -1 plot shows many examples of spectral-intersection confusion (e.g., the dispersed arm of another source overlapping the extracted source at the wavelength expected by ACIS order sorting). These are denoted with slanted lines at approximately -6.5 and -9 A. 

#### The format of this plot is difficult to use to assess arm confusion so lets look at the data now but with the y-axis as the ACIS-resolved CCD energy.

In [ ]:

#REDO the filtering with a smaller tg_d window to remove events outside default HETG extraction window
meg_p1 = filter_evt_crate(evt_data, 1, 2, 4.6E-4)
meg_m1 = filter_evt_crate(evt_data,-1, 2, 4.6E-4)
heg_p1 = filter_evt_crate(evt_data, 1, 1, 4.6E-4)
heg_m1 = filter_evt_crate(evt_data, -1, 1, 4.6E-4)

#display all events assigned to orders -3,-2,-1,+1,+2,+3
all_evts_meg_filt = np.where( (np.abs(evt_data.tg_m.values) <= 3) & (evt_data.tg_part.values == 2) )
all_evts_heg_filt = np.where( (np.abs(evt_data.tg_m.values) <= 3 ) & (evt_data.tg_part.values == 1) )


# Create a 2×2 figure (4 sub‑plots)
fig, ax = plt.subplots(
    nrows=1, ncols=2,                # 2 rows, 2 columns
    figsize=(16, 5),                # optional: figure size in inches
)

ax[0].scatter(tg_mlam[all_evts_meg_filt], energy[all_evts_meg_filt], s=1, color='grey')
ax[0].scatter(tg_mlam[meg_m1], energy[meg_m1], s=1, color='green')
ax[0].scatter(tg_mlam[meg_p1], energy[meg_p1], s=1, color='blue')
ax[0].set_title('MEG -1 and +1')


ax[1].scatter(tg_mlam[all_evts_heg_filt], energy[all_evts_heg_filt], s=1, color='grey')
ax[1].scatter(tg_mlam[heg_m1], energy[heg_m1], s=1, color='green')
ax[1].scatter(tg_mlam[heg_p1], energy[heg_p1], s=1, color='blue')
ax[1].set_title('HEG -1 and +1')

for i in ax:
    i.set_ylim(300,7500)
    i.set_xlabel('m * Wavelength [Angstrom]')
    i.set_ylabel('ACIS Energy [eV]')

#### The above plots are the so-called HETG 'banana plots' and help reveal the extent to which a source's arm/order suffers from confusion. The HETG gratings equation tells us the wavelength of every event based on its position along the dispersion direction from the 0th order. This wavelength (energy) is then compared to its ACIS CCD-determined energy to ensure it is compatible with the gratings-derived energy (wavelength). Only events whose ACIS CCD energy is consistent with the gratings-determined energy are ultimately assigned to the extracted spectrum with standard CIAO tools. Shown here in gray are all events assigned to any of the -3, -2, -1, +1, +2, +3 orders. The green and blue events are those that fall within the 4.6E-4 cross dispersion direction denoted by the yellow bars in the previous plots. 

#### Looking closer at the MEG+1 portion of the spectrum you can see the previously discussed 'point source confusion' at \~9 A. Note how it is a vertical bar in all three positive orders because the 0th order contains events over a much larger portion of ACIS CCD energy compared to the wavelength space at 9 A. Also note the curved feature starting at \~13 A in the MEG+3 order (grey) and bending ultimately towards the MEG+1 order at \~ 20 A. This is a clear indication of arm confusion from a source somewhere else on the detector but in line with the dispersed spectrum of our extracted source. The confusing source would likely contaminate all portions of the MEG+1 spectrum >\~ 20.

#### CrissCross and its helper tool clean_spec facilitate the analysis of spectra from crowded HETG fields. The next and final step would be to fit the spectra with a model and learn somethign about the underlying astrophysics. This is beyond the scope of this tutorial but there are other CIAO tools that help with this process.

## (2) Run CrissCross for a single source on multiple obsIDs and combine the spectra

#### Most HETG observations are campaigns with multiple obsIDs. CrissCross excels at running over a large number of observations and producing confusion tables for multiple source in every obsID. Users should see the crisscross and clean_spec ahelp files for more examples of how this is done. Here we show how to assess confusion for a single source over multiple obsIDs. 

#### Let's download two more Orion Nebula Cluster pre-main sequence star observations and re-process them so chandra_repro extracts spectra of COUP 394.

In [ ]:
#NOTE -- we will still use obsID 3 from above but intentionally not re-download it here.
obsid_list = [22999, 23000]

for i in obsid_list:
    os.system(f'download_chandra_obsid {i}')

In [ ]:
#run chandra repro using the zeroth order source location for COUP 394

for i in obsid_list:
    chandra_repro.punlearn()
    chandra_repro.indir = i
    chandra_repro.root = src_root
    chandra_repro.tg_zo_position = f'{src_RA},{src_DEC}'
    chandra_repro.clobber=False

    print(chandra_repro)

    a = chandra_repro()

#### Instead of running CrissCross with a single evt2 file, here we will use a list of the three obsIDs that we have downloaded. They are identfied with glob.

In [ ]:
#create a list of evt2 files
evt2_list = glob.glob(f'*/repro/{src_root}_repro_evt2.fits')
print(evt2_list)

#### We will save the CrissCross output to a new directory.

In [ ]:
run_crisscross(evt2_file=evt2_list, cc_outdir='cc_tutorial_multiobs', main_list='input_files/full_coup_src_list.tsv',  single_src_pos=f'{src_RA},{src_DEC}', single_src_root=src_root, clobber_par=False)

#### To run clean_spec on multiple pha2 files we need to provide a list of pha2_files and the relevant crisscross confusion table files. We will save the cleaned spectra to a new directory.

In [ ]:
pha2_list = sorted(glob.glob(f'*/repro/{src_root}_repro_pha2.fits'))
cc_table_list = sorted(glob.glob(f'cc_tutorial_multiobs/*/*/table_fits_data/*.fits'))

spec_dir = 'coup394_multiobs_spec'
os.makedirs(spec_dir ,exist_ok=True)

for i in range(0,len(pha2_list)):
    clean_spec(cc_table=cc_table_list[i], pha_file = pha2_list[i], spec_root=f'{spec_dir}/{src_root}', arf_file=None, resp_dir=None, clobber=False)

#### Now that we have a set of cleaned spectra from three obsIDs, lets combine them with the CIAO tool combine_grating_spectra. This is useful for visualization purposes because it can combine multiple spectra of the same source from different obsIDs. It can also combine the +1-1 orders of the same arm. It should be noted that it is typically not best practice to fit combined spectra. Instead, users are encouraged to perform joint-fits on the individual spectra and use the combining for visualization purposes. Moreover, caution should be taken when combining spectra from sources that undergo significant variability.

#### Let's start by combining the original spectra from the chandra_repro reprocessing. 

In [ ]:
#make a directory to hold the combine_grating_spectra combined files.
comb_spec = 'combined_spec'
os.makedirs(comb_spec, exist_ok=True)

#combine the original 'confused' spectra
combine_grating_spectra.punlearn()
combine_grating_spectra.infile=f'*/repro/{src_root}*_pha2.fits'
combine_grating_spectra.arf=f'*/repro/tg/*.arf'
combine_grating_spectra.rmf=f'*/repro/tg/*.rmf'
combine_grating_spectra.outroot=f'{comb_spec}/{src_root}_original'
combine_grating_spectra.add_plusminus='yes'
combine_grating_spectra.order=1
combine_grating_spectra.clobber=False

print(combine_grating_spectra)

a = combine_grating_spectra()

#### Unfortunately, combine_grating_spectra looks for rmfs in the same directory structure as the PHA2 files so we will have to copy the original RMFs to the new 'cleaned' directory to combine all the cleaned files and responses. This is typically not a big deal but RMFs can be relatively large files so if you are working with many observations then disk space should be considered.

In [ ]:
#ID the original RMFs and make a copy in the spec_dir
rmf_list = sorted(glob.glob(f'*/repro/tg/*.rmf'))
print(rmf_list)

for i in rmf_list:
    shutil.copy2(i, spec_dir)

In [ ]:
#combine the cleaned spectra
combine_grating_spectra.punlearn()
combine_grating_spectra.infile=f'{spec_dir}/*.pha2'
combine_grating_spectra.arf=f'{spec_dir}/*.arf'
combine_grating_spectra.rmf=f'{spec_dir}/*.rmf'
combine_grating_spectra.outroot=f'{comb_spec}/{src_root}_cleaned'
combine_grating_spectra.add_plusminus='yes'
combine_grating_spectra.order=1
combine_grating_spectra.clobber=False

print(combine_grating_spectra)

a = combine_grating_spectra()


#### We can load the COUP 394 spectra combined from three obsIDs into sherpa and plot the comparison of the confused vs cleaned spectra again. 

In [ ]:
orig_heg = f'{comb_spec}/COUP_394_original_combo_heg_abs1.pha'
orig_meg = f'{comb_spec}/COUP_394_original_combo_meg_abs1.pha'
cleaned_heg = f'{comb_spec}/COUP_394_cleaned_combo_heg_abs1.pha'
cleaned_meg = f'{comb_spec}/COUP_394_cleaned_combo_meg_abs1.pha'

#run clean so we can override the previous example's sherpa environment
clean()
load_data(1,orig_heg, use_errors=True)
load_data(2,orig_meg, use_errors=True)
load_data(3,cleaned_heg, use_errors=True)
load_data(4,cleaned_meg, use_errors=True)

In [ ]:
#the sherpa ids of the combined datasets
combined_specs = [1,2,3,4]

#set the analysis to wavelength so we can plot in Angstrom
for i in combined_specs:
    set_analysis(i, 'wave')

#bin the counts up for visualization purposes
bin_num_comb=3
for i in combined_specs:
    group_width(i, bin_num_comb)

#setup the combined arm labels for plotting
arm_label_comb = ['three obsIDs +1/-1 combined heg','three obsIDs +1/-1 combined meg']

plt.figure(figsize=(9, 5))

#plot the original (confused) HEG and MEG spectra in red
plot("data", 1, "data", 2, color='red',label=['confused','confused'], yerrorbars=False, linestyle='-', marker=None)

#overplot the new 'cleaned' HEG and MEG spectra in blue
plot("data", 3, "data", 4, color='blue',label=['cleaned','cleaned'], overplot=True, yerrorbars=False, linestyle='-', marker=None)

# set the same x-limits for all panels:
axes = plt.gcf().axes
for ax, arm in zip(axes,arm_label_comb):
    ax.set_xlim(0,20)
    ax.set_title(arm)

#label the figure with color
fig = plt.gcf()
fig.text(0.30, 0.96, "Confused",  ha='left',  va='center', color='red',  fontsize=16)
fig.text(0.425, .96, "vs",    va='center', color='black',  fontsize=16)
fig.text(0.565, .96, "Cleaned", ha='right', va='center', color='blue', fontsize=16)
plt.draw()
plt.subplots_adjust(left=0, hspace=0.5, wspace=0.3)

In [ ]:

arm_label_comb_arf = ['three obsIDs +1/-1 combined heg response','three obsIDs +1/-1 combined meg response']

plt.figure(figsize=(9, 5))
#original ARFs
plot("arf", 1, "arf", 2, label=['confused','confused'], color='red')
#cleaned ARFs
plot("arf", 3, "arf", 4, label=['cleaned','cleaned'],color='blue', overplot=True)

axes = plt.gcf().axes
for ax, arm in zip(axes,arm_label_comb_arf):
    ax.set_xlim(0, 20)
    ax.set_title(arm)

plt.subplots_adjust(left=0, hspace=0.5, wspace=0.3)

#### As you increase exposure time with multiple obsIDs, more emission lines are visible for COUP 394. Also note how the **combined** response does not go to zero because confusion does not occur at the same spot in all obsIDs due to roll angle variations and source variability. Multiple observations can be used to mitigate confusion in crowded regions.This comparison of the original (confused) data with the cleaned data demonstrate the necessity to account for confusion from other sources in the field of view.  

#### For questions, comments or suggestions about CrissCross and its functionality, please contact the CXC helpdesk or principe@space.mit.edu.